<a href="https://colab.research.google.com/github/Nathruth/pothole_detection/blob/master/models/capstone2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
files.upload()


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"nathruth","key":"0dfb029932ae5af0c086cc4a18f5a9e0"}'}

In [2]:
!ls -l kaggle.json


-rw-r--r-- 1 root root 64 Jan 19 16:31 kaggle.json


In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [4]:
!kaggle datasets list


ref                                                                title                                                    size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-----------------------------------------------------------------  -------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
neurocipher/heartdisease                                           Heart Disease                                            3491  2025-12-11 15:29:14.327000           2114        318  1.0              
mabubakrsiddiq/retail-store-product-sales-simulation-dataset       🏪 Retail Store Product Sales Simulation Dataset       1383545  2026-01-16 13:12:07.310000              0         27  1.0              
saidaminsaidaxmadov/chocolate-sales                                Chocolate Sales                                        468320  2026-01-04 14:23:35.490000              0         59  1.0     

In [5]:
!pip install kaggle --quiet
!pip install torch torchvision --quiet


In [6]:
!kaggle datasets download -d atulyakumar98/pothole-detection-dataset


Dataset URL: https://www.kaggle.com/datasets/atulyakumar98/pothole-detection-dataset
License(s): CC0-1.0
 58% 112M/194M [00:00<00:00, 1.11GB/s]
100% 194M/194M [00:00<00:00, 590MB/s] 


In [7]:
!mkdir -p ./data
!unzip pothole-detection-dataset.zip -d ./data


Archive:  pothole-detection-dataset.zip
  inflating: ./data/normal/1.jpg     
  inflating: ./data/normal/10.jpg    
  inflating: ./data/normal/100.jpg   
  inflating: ./data/normal/101.jpg   
  inflating: ./data/normal/102.jpg   
  inflating: ./data/normal/103.jpg   
  inflating: ./data/normal/104.jpg   
  inflating: ./data/normal/105.jpg   
  inflating: ./data/normal/106.jpg   
  inflating: ./data/normal/107.jpg   
  inflating: ./data/normal/108.jpg   
  inflating: ./data/normal/109.jpg   
  inflating: ./data/normal/11.jpg    
  inflating: ./data/normal/110.jpg   
  inflating: ./data/normal/111.jpg   
  inflating: ./data/normal/112.jpg   
  inflating: ./data/normal/113.jpg   
  inflating: ./data/normal/114.jpg   
  inflating: ./data/normal/115.jpg   
  inflating: ./data/normal/116.jpg   
  inflating: ./data/normal/117.jpg   
  inflating: ./data/normal/118.jpg   
  inflating: ./data/normal/119.jpg   
  inflating: ./data/normal/12.jpg    
  inflating: ./data/normal/120.jpg   
  inflatin

In [8]:
import os
os.listdir("./data")


['normal', 'potholes']

In [9]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# Define folder paths
normal_dir = "./data/normal"
potholes_dir = "./data/potholes"

# Collect image paths
normal_files = [os.path.join(normal_dir, f) for f in os.listdir(normal_dir) if f.endswith(".jpg")]
potholes_files = [os.path.join(potholes_dir, f) for f in os.listdir(potholes_dir) if f.endswith(".jpg")]

# Create DataFrames
df_normal = pd.DataFrame({"path": normal_files, "label": 0})
df_potholes = pd.DataFrame({"path": potholes_files, "label": 1})

df = pd.concat([df_normal, df_potholes], ignore_index=True)

# Train / val / test split
df_train, df_temp = train_test_split(df, test_size=0.3, stratify=df["label"], random_state=42)
df_val, df_test   = train_test_split(df_temp, test_size=0.5, stratify=df_temp["label"], random_state=42)

# Make splits folder
os.makedirs("./data/splits", exist_ok=True)

# Save CSVs
df_train.to_csv("./data/splits/train.csv", index=False)
df_val.to_csv("./data/splits/val.csv", index=False)
df_test.to_csv("./data/splits/test.csv", index=False)

# Load CSVs into DataFrames
df_train = pd.read_csv("./data/splits/train.csv")
df_val   = pd.read_csv("./data/splits/val.csv")
df_test  = pd.read_csv("./data/splits/test.csv")

print("Train:", len(df_train), "Val:", len(df_val), "Test:", len(df_test))



Train: 476 Val: 102 Test: 103


In [10]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])


In [11]:
from torch.utils.data import Dataset
from PIL import Image

class PotholeDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]["path"]
        label = self.df.iloc[idx]["label"]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


In [12]:
from torch.utils.data import DataLoader

batch_size = 16

train_dataset = PotholeDataset(df_train, transform=transform)
val_dataset = PotholeDataset(df_val, transform=transform)
test_dataset = PotholeDataset(df_test, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")


Train batches: 30, Val batches: 7, Test batches: 7


In [13]:
import torch
import torch.nn as nn
import torchvision.models as models

device = "cuda" if torch.cuda.is_available() else "cpu"

mobilenet = models.mobilenet_v2(pretrained=True)

# Freeze all layers
for param in mobilenet.parameters():
    param.requires_grad = False

# Replace the classifier for 2 classes (normal / pothole)
mobilenet.classifier[1] = nn.Linear(mobilenet.last_channel, 2)

mobilenet.to(device)
print(mobilenet)


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 48.3MB/s]


MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mobilenet.classifier.parameters(), lr=1e-3)

num_epochs = 5

# -------- Training loop --------
history = []

for epoch in range(num_epochs):
    mobilenet.train()
    running_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = mobilenet(images)
        loss = nn.CrossEntropyLoss()(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # Validation
    mobilenet.eval()
    val_preds = []
    val_labels = []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = mobilenet(images)
            preds = torch.argmax(outputs, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(val_labels, val_preds)
    f1 = f1_score(val_labels, val_preds)
    cm = confusion_matrix(val_labels, val_preds)

    # Append epoch metrics to history
    history.append({
        "epoch": epoch+1,
        "loss": running_loss/len(train_loader),
        "val_accuracy": acc,
        "val_f1": f1,
        "val_confusion_matrix": cm.tolist()  # store as list
    })

    print(f"Epoch {epoch+1} → Loss: {running_loss/len(train_loader):.4f}, Val Acc: {acc:.3f}, Val F1: {f1:.3f}")


Epoch 1 → Loss: 0.4247, Val Acc: 0.941, Val F1: 0.942
Epoch 2 → Loss: 0.1823, Val Acc: 0.980, Val F1: 0.980
Epoch 3 → Loss: 0.1570, Val Acc: 0.980, Val F1: 0.980
Epoch 4 → Loss: 0.1331, Val Acc: 0.980, Val F1: 0.980
Epoch 5 → Loss: 0.1569, Val Acc: 0.931, Val F1: 0.933


In [15]:
# -------- Test set evaluation --------
mobilenet.eval()
test_preds = []
test_labels = []
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = mobilenet(images)
        preds = torch.argmax(outputs, dim=1)
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

test_acc = accuracy_score(test_labels, test_preds)
test_f1 = f1_score(test_labels, test_preds)
test_cm = confusion_matrix(test_labels, test_preds)

print("\n--- TEST SET RESULTS ---")
print(f"Test Accuracy: {test_acc:.3f}, F1 (potholes): {test_f1:.3f}")
print("Confusion Matrix:")
print(test_cm)



--- TEST SET RESULTS ---
Test Accuracy: 0.961, F1 (potholes): 0.962
Confusion Matrix:
[[49  4]
 [ 0 50]]


In [16]:
df_history = pd.DataFrame(history)
df_history.to_csv("experiment_history.csv", index=False)


In [17]:
from google.colab import files

files.download("experiment_history.csv")  # or experiment_history.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
#  Unfreeze last few layers for fine-tuning
for name, param in mobilenet.features.named_parameters():
    param.requires_grad = False  # freeze all by default

# Unfreeze last 3 MobileNetV2 blocks
for name, param in list(mobilenet.features.named_parameters())[-30:]:
    param.requires_grad = True


optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, mobilenet.parameters()), lr=1e-4)

from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])


train_dataset.transform = train_transform


num_epochs = 5

for epoch in range(num_epochs):
    mobilenet.train()
    running_loss = 0
    for images, labels in tqdm(train_loader, desc=f"Fine-tune Epoch {epoch+1}/{num_epochs}"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = mobilenet(images)
        loss = nn.CrossEntropyLoss()(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # Validation
    mobilenet.eval()
    val_preds = []
    val_labels = []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = mobilenet(images)
            preds = torch.argmax(outputs, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(val_labels, val_preds)
    f1 = f1_score(val_labels, val_preds)
    cm = confusion_matrix(val_labels, val_preds)

    print(f"\nEpoch {epoch+1}/{num_epochs} → Loss: {running_loss/len(train_loader):.4f}")
    print(f"Val Accuracy: {acc:.3f}, F1 (potholes): {f1:.3f}")
    print("Confusion Matrix:")
    print(cm)


Fine-tune Epoch 1/5: 100%|██████████| 30/30 [00:13<00:00,  2.22it/s]



Epoch 1/5 → Loss: 0.1468
Val Accuracy: 0.971, F1 (potholes): 0.970
Confusion Matrix:
[[51  2]
 [ 1 48]]


Fine-tune Epoch 2/5: 100%|██████████| 30/30 [00:13<00:00,  2.24it/s]



Epoch 2/5 → Loss: 0.0374
Val Accuracy: 0.980, F1 (potholes): 0.980
Confusion Matrix:
[[51  2]
 [ 0 49]]


Fine-tune Epoch 3/5: 100%|██████████| 30/30 [00:13<00:00,  2.25it/s]



Epoch 3/5 → Loss: 0.0231
Val Accuracy: 0.980, F1 (potholes): 0.980
Confusion Matrix:
[[51  2]
 [ 0 49]]


Fine-tune Epoch 4/5: 100%|██████████| 30/30 [00:13<00:00,  2.19it/s]



Epoch 4/5 → Loss: 0.0153
Val Accuracy: 0.971, F1 (potholes): 0.970
Confusion Matrix:
[[51  2]
 [ 1 48]]


Fine-tune Epoch 5/5: 100%|██████████| 30/30 [00:13<00:00,  2.25it/s]



Epoch 5/5 → Loss: 0.0355
Val Accuracy: 0.980, F1 (potholes): 0.980
Confusion Matrix:
[[51  2]
 [ 0 49]]


In [19]:
mobilenet.eval()


MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [20]:
!pip install onnx --upgrade


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 79.7 MB/s eta 0:00:00


In [21]:
!pip install onnxscript --upgrade


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 kB 17.4 MB/s eta 0:00:00


In [22]:
import torch

mobilenet.eval()
mobilenet.cpu()


MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [25]:
dummy_input = torch.randn(1, 3, 224, 224)

torch.onnx.export(
    mobilenet,
    dummy_input,
    "mobilenetv2_pothole.onnx",
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
)


print("ONNX export finished")


W0119 16:43:29.003000 255 torch/onnx/_internal/exporter/_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 127, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 122, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /github/workspace/onnx/version_converter/adapters/axes_input_to_attribute.h:65: adapt: Asserti

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 104 of general pattern rewrite rules.
ONNX export finished


In [26]:
import onnx

model = onnx.load("mobilenetv2_pothole.onnx")
onnx.save(
    model,
    "mobilenetv2_pothole_single.onnx",
    save_as_external_data=False
)


In [27]:
!ls -lh mobilenetv2_pothole_single.onnx
!ls mobilenetv2_pothole_single.onnx.data



-rw-r--r-- 1 root root 8.7M Jan 19 16:44 mobilenetv2_pothole_single.onnx
ls: cannot access 'mobilenetv2_pothole_single.onnx.data': No such file or directory


In [ ]:
import json

metadata = {
    "classes": ["normal", "potholes"],
    "input_size": [3, 224, 224],
    "mean": [0.485, 0.456, 0.406],
    "std": [0.229, 0.224, 0.225]
}

with open("metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)


In [28]:
from google.colab import files

files.download("mobilenetv2_pothole_single.onnx")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>